## Script para concatenar planilhas de dados de estação por estado

##### Esse código serve para unir planilhas de dados de estações por estado, quando a UF fornece um tipo de dado, e internamente coletamos outras informações sobre a mesma UF

#### Imports 

In [1]:
import pandas as pd
from pathlib import Path
import os
import numpy as np
import io
import msal
from office365.graph_client import GraphClient
import json
import re
import requests, base64

In [23]:
!pip install msal

In [33]:
%pip install -U Office365-REST-Python-Client msal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


#### Importar files quando acessados direto pelo OneDrive

In [4]:
TENANT = "common"   # or your tenant ID like "contoso.onmicrosoft.com"
CLIENT_ID = "b79b4905-c2a4-4f94-8bdb-7107be5d2864"
SCOPES = ["Files.Read"]              

def acquire_token():
    app = msal.PublicClientApplication(
        CLIENT_ID,
        authority=f"https://login.microsoftonline.com/{TENANT}"
    )
    flow = app.initiate_device_flow(scopes=SCOPES)
    if "user_code" not in flow:
        raise RuntimeError("Device flow init failed:\n" + json.dumps(flow, indent=2))
    print(flow["message"])           # open the URL and enter the code
    result = app.acquire_token_by_device_flow(flow)
    if "access_token" not in result:
        raise RuntimeError("Token acquisition failed:\n" + json.dumps(result, indent=2))
    return {"access_token": result["access_token"]}

#client = GraphClient(acquire_token)

#for it in client.me.drive.root.children.get().execute_query():
#    print(it.name)

# assumes: client = GraphClient(acquire_token) already works

# get a bearer once
token = acquire_token()["access_token"]
H = {"Authorization": f"Bearer {token}"}

# 1) Find the shared item "GAr_BR"
url = "https://graph.microsoft.com/v1.0/me/drive/sharedWithMe?$select=name,webUrl,remoteItem&$top=200"
items = []
while url:
    data = requests.get(url, headers=H).json()
    items += data.get("value", [])
    url = data.get("@odata.nextLink")

gar = next((x for x in items if x.get("name", "").lower() == "gar_br"), None)
if not gar:
    raise RuntimeError("GAr_BR not found in Shared with me")

# 2) Resolve the owner drive and item id
remote = gar.get("remoteItem", {}) or {}
drive_id = remote.get("parentReference", {}).get("driveId")
item_id  = remote.get("id")

# fallback via the share link if needed
if not drive_id or not item_id:
    share_url = gar.get("webUrl")
    enc = base64.urlsafe_b64encode(share_url.encode()).decode().rstrip("=")
    info = requests.get(f"https://graph.microsoft.com/v1.0/shares/u!{enc}/driveItem", headers=H).json()
    drive_id = info["parentReference"]["driveId"]
    item_id  = info["id"]

# helpers
def children(pid):
    u = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{pid}/children?$select=id,name,folder&$top=999"
    out = []
    while u:
        d = requests.get(u, headers=H).json()
        out += d.get("value", [])
        u = d.get("@odata.nextLink")
    return out

def pick(items, name):
    key = name.lower()
    return next((v for v in items if "folder" in v and v["name"].lower() == key), None)

To sign in, use a web browser to open the page https://microsoft.com/devicelogin and enter the code E7AB55QJV to authenticate.


In [118]:
# Alterar informações conforme desejado
obj = "Objetivo_07"
year = "2025" # ALterar ano quando necessário
data = "dados_monitoramento"
uf = "RJ" # alterar UF quando necessário

In [119]:
# Listar files dentro de GAr_BR e fazer drill down
lvl1 = children(item_id)
print("Inside GAr_BR:", [v["name"] for v in lvl1])

obj = pick(lvl1, obj)
lvl2 = children(obj["id"])
yr  = pick(lvl2, year)
lvl3 = children(yr["id"])
ds  = pick(lvl3, data)
lvl4 = children(ds["id"])
ufn = pick(lvl4, uf)

final = children(ufn["id"])
for v in final:
    print(("DIR" if "folder" in v else "FILE"), v["name"])

Inside GAr_BR: ['Apresentações', 'Dados', 'Gestão', 'Objetivo_01', 'Objetivo_02', 'Objetivo_03', 'Objetivo_04', 'Objetivo_05', 'Objetivo_06', 'Objetivo_07', 'Referências', 'readme.txt']
DIR Dados Coletados
FILE INEA_2024_Automatica.csv
FILE INEA_2024_Semiautomatica.csv


##### a) Rio de Janeiro

In [120]:
def if_folder(folder_name):
    lvl = children(ufn["id"])
    dc = pick(lvl, folder_name) or pick(lvl, folder_name.replace("_", " "))
    if not dc:
        raise RuntimeError(f"Folder '{folder_name}' not found. Here: {[v['name'] for v in lvl if 'folder' in v]}")
    years = [v for v in children(dc["id"]) if "folder" in v and re.fullmatch(r"\d{4}", v["name"])]
    return sorted(years, key=lambda v: v["name"])

def fetch_head_csv(file_id, bytes_max=131072):
    url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{file_id}/content"
    r = requests.get(url, headers={**H, "Range": f"bytes=0-{bytes_max-1}"}, stream=True)
    data = b""
    for chunk in r.iter_content(65536):
        data += chunk
        if len(data) >= bytes_max:
            break
    return data

def sample_file(item, converters=None):
    name = item["name"].lower()
    url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{item['id']}/content"
    if name.endswith(".csv"):
        buf = fetch_head_csv(item["id"])
        df = pd.read_csv(io.BytesIO(buf), sep=None, engine="python", nrows=200, low_memory=False,
                         converters=converters)
    elif name.endswith((".xlsx", ".xls")):
        bin_ = requests.get(url, headers=H).content  # full download for Excel
        df = pd.read_excel(io.BytesIO(bin_), sheet_name=0, nrows=50)
    else:
        return None
    cols = [f"{c}<{df[c].dtype}>" for c in df.columns]
    return cols

def iterate_years(folder_name, types):
    if isinstance(types, str):
        types = (types.lower(),)
    else:
        types = tuple(t.lower() for t in types)

    year_nodes = if_folder(folder_name)
    for y in year_nodes:
        items = children(y["id"])
        files = [v for v in items if "folder" not in v and any(v["name"].lower().endswith(ext) for ext in types)]
        print(f"\nYear {y['name']} — {len(files)} file(s)")

In [121]:
folder_name = 'Dados Coletados'
types=(".csv", ".xlsx")
iterate_years(folder_name, types)


Year 2016 — 24 file(s)

Year 2017 — 24 file(s)

Year 2018 — 24 file(s)

Year 2019 — 25 file(s)

Year 2020 — 24 file(s)

Year 2021 — 24 file(s)

Year 2022 — 22 file(s)

Year 2023 — 24 file(s)

Year 2024 — 24 file(s)


In [122]:
# Características dos dados coletados do RJ
FLAG_MAP = {
    "??":"Valor Suspeito",
    "?A":"Suspeita Automática",
    "<":"Abaixo do Limite de Detecção",
    ">":"Acima do Limite de Detecção",
    "I":"Inexistente",
    "IA":"Amostra Insuficiente / Invalidação Automática",
    "IE":"Invalidado pelo Equipamento",
    "IL":"Interferência Local",
    "IP":"Falta de Energia Elétrica",
    "IR":"Fora da Faixa de Medição",
    "IU":"Invalidado pelo Usuário",
    "NR":"Não Requisitado",
    "PP":"Processo Parado",
    "V":"Válido",
    "VC":"Valor Calculado por Fórmula",
    "VI":"Inserido/Editado pelo Usuário",
    "VU":"Validado pelo Usuário",
}

# Normalização dos meses 
_MONTHS = {"jan":"jan","fev":"feb","mar":"mar","abr":"apr","mai":"may","jun":"jun",
           "jul":"jul","ago":"aug","set":"sep","out":"oct","nov":"nov","dez":"dec"}

_MONTH_RE = re.compile(r"-(jan|fev|mar|abr|mai|jun|jul|ago|set|out|nov|dez)-", re.I)

In [135]:
import unicodedata
from zipfile import BadZipFile
from warnings import filterwarnings
filterwarnings("ignore", message="Workbook contains no default style")

target_year = "2018"   # change this

In [139]:
def inorm(s: str):
    s = unicodedata.normalize("NFKD", s).encode("ASCII", "ignore").decode()
    return s.lower().replace(" ", "").replace("_", "").replace("-", "")

# Portuguese month handling
_MONTHS = {"jan":"jan","fev":"feb","mar":"mar","abr":"apr","mai":"may","jun":"jun",
           "jul":"jul","ago":"aug","set":"sep","out":"oct","nov":"nov","dez":"dec"}
_MONTH_RE = re.compile(r"-(jan|fev|mar|abr|mai|jun|jul|ago|set|out|nov|dez)-", re.I)
def _pt_months_to_en(s: str) -> str:
    if not isinstance(s, str): s = str(s)
    return _MONTH_RE.sub(lambda m: f"-{_MONTHS[m.group(1).lower()]}-", s)

def _first_last_dt_common(series_str: pd.Series):
    # try strict formats first, then a safe fallback
    attempts = [
        ("%Y-%m-%d %H:%M:%S", False),
        ("%Y-%m-%d %H:%M",     False),
        ("%d-%b-%Y %H:%M:%S",  True),
        ("%d-%b-%Y %H:%M",     True),
        ("%d/%m/%Y %H:%M:%S",  True),
        ("%d/%m/%Y %H:%M",     True),
    ]
    dt = pd.Series(pd.NaT, index=series_str.index)
    for fmt, dayfirst in attempts:
        m = dt.isna()
        if not m.any(): break
        dt.loc[m] = pd.to_datetime(series_str[m], format=fmt, errors="coerce", dayfirst=dayfirst)
    if dt.isna().any():
        m = dt.isna()
        dt.loc[m] = pd.to_datetime(series_str[m], dayfirst=True, errors="coerce")
    return (dt.min(), dt.max()) if dt.notna().any() else (None, None)

# XLSX readers
def _header_map_xlsx(blob: bytes):
    hdr = pd.read_excel(io.BytesIO(blob), header=None, nrows=3, engine="openpyxl")
    stations   = hdr.iloc[1].ffill().astype(str).tolist()
    pollutants = hdr.iloc[2].astype(str).tolist()
    by_station = {}
    for j, (st, pol) in enumerate(zip(stations, pollutants)):
        if j == 0:  # "Data e Hora"
            continue
        if re.search(r"qa/?qc", str(pol), flags=re.I):
            continue
        by_station.setdefault(st.strip(), set()).add(str(pol).strip())
    return by_station

def _first_last_dt_xlsx(blob: bytes):
    col0 = pd.read_excel(io.BytesIO(blob), header=None, usecols=[0], skiprows=3,
                         engine="openpyxl", dtype=str).iloc[:,0].dropna().str.strip()
    return _first_last_dt_common(col0.map(_pt_months_to_en))

# XLS (BIFF) readers
def _header_map_xls(blob: bytes):
    hdr = pd.read_excel(io.BytesIO(blob), header=None, nrows=3, engine="xlrd")
    stations   = hdr.iloc[1].ffill().astype(str).tolist()
    pollutants = hdr.iloc[2].astype(str).tolist()
    by_station = {}
    for j, (st, pol) in enumerate(zip(stations, pollutants)):
        if j == 0:
            continue
        if re.search(r"qa/?qc", str(pol), flags=re.I):
            continue
        by_station.setdefault(st.strip(), set()).add(str(pol).strip())
    return by_station

def _first_last_dt_xls(blob: bytes):
    col0 = pd.read_excel(io.BytesIO(blob), header=None, usecols=[0], skiprows=3,
                         engine="xlrd", dtype=str).iloc[:,0].dropna().str.strip()
    return _first_last_dt_common(col0.map(_pt_months_to_en))

def download(item):
    url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{item['id']}/content"
    r = requests.get(url, headers=H, allow_redirects=True)
    r.raise_for_status()
    return r.content, r.headers.get("Content-Type","").lower()

# --- find "Dados Coletados" under the UF folder ---
lvl = children(ufn["id"])
dc = next((v for v in lvl if "folder" in v and inorm(v["name"]) in {inorm("Dados Coletados"), inorm("Dados_Coletados")}), None)
if not dc:
    raise RuntimeError(f"'Dados Coletados' not found. Here: {[v['name'] for v in lvl if 'folder' in v]}")

# pick the single year
lvl_dc = children(dc["id"])
y = next((v for v in lvl_dc if "folder" in v and v["name"] == target_year), None)
if not y:
    raise RuntimeError(f"Year {target_year} not found in '{dc['name']}'. Available: {[v['name'] for v in lvl_dc if 'folder' in v]}")

# auto = files at year root; manual = files under "Estações Manuais"
y_items = children(y["id"])
man_folder = next((v for v in y_items if "folder" in v and inorm(v["name"]) == inorm("Estações Manuais")), None)

auto_files = [v for v in y_items if "folder" not in v and v["name"].lower().endswith((".xlsx",".xls"))]
manual_files = []
if man_folder:
    man_items = children(man_folder["id"])
    manual_files = [v for v in man_items if "folder" not in v and v["name"].lower().endswith((".xlsx",".xls"))]

print(f"Year {target_year} - Files Auto: {len(auto_files)} - Files Manual: {len(manual_files)}")

# --- process files and build df_year_summary ---
rows = []
for mode, files in [("automatic", auto_files), ("manual", manual_files)]:
    for f in files:
        blob, ctype = download(f)
        ext = os.path.splitext(f["name"])[1].lower()
        try:
            if ext == ".xlsx" or "spreadsheetml" in ctype:
                by_station = _header_map_xlsx(blob)
                first_dt, last_dt = _first_last_dt_xlsx(blob)
            elif ext == ".xls" or "application/vnd.ms-excel" in ctype:
                by_station = _header_map_xls(blob)
                first_dt, last_dt = _first_last_dt_xls(blob)
            else:
                print(f"skip {f['name']} (ext={ext}, type={ctype})")
                continue
        except BadZipFile:
            print(f"{f['name']}: not a valid .xlsx (BadZipFile). If this is .xls, rename to .xls or check file.")
            continue

        for st, pols in by_station.items():
            rows.append({
                "year": target_year,
                "mode": mode,
                "file": f["name"],
                "station": st,
                "pollutants": sorted(pols),
                "first_ts": first_dt,
                "last_ts": last_dt,
            })

df_year_summary = pd.DataFrame(rows).sort_values(["mode","station","file"]).reset_index(drop=True)

# quick sanity: counts per mode and station coverage
print("\nStations per mode:")
print(df_year_summary.groupby("mode")["station"].nunique())

Year 2018 - Files Auto: 24 - Files Manual: 25

Stations per mode:
mode
automatic     50
manual       100
Name: station, dtype: int64


In [140]:
df_year_summary.head()

,year,mode,file,station,pollutants,first_ts,last_ts
0,2018,automatic,2018_medio_paraiba_1.xlsx,BM - Boa Sorte,"[Partículas Inaláveis (<10µm) [µg/m³], Partícu...",2018-01-01,2018-12-31 23:00:00
1,2018,automatic,2018_medio_paraiba_1.xlsx,BM - Bocaininha,"[Partículas Inaláveis (<10µm) [µg/m³], Partícu...",2018-01-01,2018-12-31 23:00:00
2,2018,automatic,2018_medio_paraiba_1.xlsx,BM - Roberto Silveira,"[Partículas Inaláveis (<10µm) [µg/m³], Partícu...",2018-01-01,2018-12-31 23:00:00
3,2018,automatic,2018_medio_paraiba_1.xlsx,BM - Sesi,"[Partículas Inaláveis (<10µm) [µg/m³], Partícu...",2018-01-01,2018-12-31 23:00:00
4,2018,automatic,2018_medio_paraiba_1.xlsx,BM - Vista Alegre,"[Partículas Inaláveis (<10µm) [µg/m³], Partícu...",2018-01-01,2018-12-31 23:00:00


In [ ]:
years = years[:1] # Testar primeiro ano

In [68]:
file_name = "INEA_2024_Automatica.csv"   
content = requests.get(f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{f['id']}/content",headers=H).content

# to pandas
df = pd.read_csv(
    io.BytesIO(content),
    low_memory=False,                 
    converters={6: str, 7: str, 9: str},
    sep=';'
)
df.head()

,Ano,Mes,Dia,Hora,Minuto,Nome da estação,Poluente,Valor,Unidade
0,2024,1,1,0,0,Cg - Val Palmas,"MP2,5","7,0",µg/m³
1,2024,1,1,1,0,Cg - Val Palmas,"MP2,5","10,0",µg/m³
2,2024,1,1,2,0,Cg - Val Palmas,"MP2,5","7,0",µg/m³
3,2024,1,1,3,0,Cg - Val Palmas,"MP2,5","11,0",µg/m³
4,2024,1,1,4,0,Cg - Val Palmas,"MP2,5","14,0",µg/m³


In [68]:
file_name = "INEA_2024_Automatica.csv"   
content = requests.get(f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{f['id']}/content",headers=H).content

# to pandas
df = pd.read_csv(
    io.BytesIO(content),
    low_memory=False,                 
    converters={6: str, 7: str, 9: str},
    sep=';'
)
df.head()

,Ano,Mes,Dia,Hora,Minuto,Nome da estação,Poluente,Valor,Unidade
0,2024,1,1,0,0,Cg - Val Palmas,"MP2,5","7,0",µg/m³
1,2024,1,1,1,0,Cg - Val Palmas,"MP2,5","10,0",µg/m³
2,2024,1,1,2,0,Cg - Val Palmas,"MP2,5","7,0",µg/m³
3,2024,1,1,3,0,Cg - Val Palmas,"MP2,5","11,0",µg/m³
4,2024,1,1,4,0,Cg - Val Palmas,"MP2,5","14,0",µg/m³


#### Importar files quando baixados no Jupyter

In [ ]:
# importar files se baixados no Jupyter
base = Path.cwd().parent  
out_dir = base / "data" / "DADOS_ESTACOES" 
out_dir.mkdir(parents=True, exist_ok=True)
#uf = 

files = os.listdir(out_dir)
for f in files:
    print(f)

#### Dados estações

In [45]:
# Alterar informações conforme desejado

obj = "Objetivo_07"
year = "2025" # Alterar ano quando necessário
data = "dados_estacoes" # Alterar tipo de informções quando necessário
uf = "RJ" 

In [46]:
# Listar files dentro de GAr_BR e fazer drill down
lvl1 = children(item_id)
print("Inside GAr_BR:", [v["name"] for v in lvl1])

obj = pick(lvl1, obj)
lvl2 = children(obj["id"])
yr  = pick(lvl2, year)
lvl3 = children(yr["id"])
ds  = pick(lvl3, data)

# Não tem level 4 nesse caso pois as pastas não são dividas por UFs; Quando for, inserir lvl4
#lvl4 = children(ds["id"]) 
#ufn = pick(lvl4, uf)

final = children(ds["id"])
for v in final:
    print(("DIR" if "folder" in v else "FILE"), v["name"])

Inside GAr_BR: ['Apresentações', 'Dados', 'Gestão', 'Objetivo_01', 'Objetivo_02', 'Objetivo_03', 'Objetivo_04', 'Objetivo_05', 'Objetivo_06', 'Objetivo_07', 'Referências', 'readme.txt']
DIR dados_estacoes_PRONTOS
DIR indicativas
DIR trash
FILE BA_estacoes.csv
FILE DF_estacoes.csv
FILE ES_estacoes.csv
FILE MA_estacoes.csv
FILE MG_estacoes.csv
FILE MT_estacoes.csv
FILE PR_estacoes.csv
FILE RJ_estacoes.csv
FILE RJ_estacoes_enviada.xlsx
FILE RS_estacoes.csv
FILE SC_estacoes.csv
FILE SP_estacoes.csv


##### a) Rio de Janeiro

In [47]:
# Neste caso queremos filtrar somente os files no folder
matches = [
    it for it in final
    if "folder" not in it
    and it["name"].upper().startswith(uf.upper())
    and it["name"].lower().endswith((".csv", ".xlsx", ".xls"))
]
matches = sorted(matches, key=lambda x: x["name"])

print(f"{len(matches)} files for {uf}:")
for it in matches:
    print(f"{it['name']}")

2 files for RJ:
RJ_estacoes.csv
RJ_estacoes_enviada.xlsx


In [48]:
from IPython.display import display

In [49]:
def download(item):
    url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{item['id']}/content"
    r = requests.get(url, headers=H, allow_redirects=True)
    r.raise_for_status()
    return r.content

dfs = {}
for it in matches:
    blob = download(it)
    name = it["name"]
    ext = os.path.splitext(name)[1].lower()

    if ext == ".csv":
        # auto-detect delimiter
        df = pd.read_csv(io.BytesIO(blob), sep=None, engine="python")
    elif ext == ".xlsx":
        df = pd.read_excel(io.BytesIO(blob), engine="openpyxl")
    elif ext == ".xls":
        # xlrd 1.2.0 is needed for .xls
        df = pd.read_excel(io.BytesIO(blob), engine="xlrd")
    else:
        print(f"Skip {name} (unsupported)")
        continue

    dfs[name] = df
    print(f"{name}: shape {df.shape}")

# preview
for name, df in dfs.items():
    print(f"\n{name}")
    display(df.head(5))

RJ_estacoes.csv: shape (121, 31)
RJ_estacoes_enviada.xlsx: shape (64, 29)

RJ_estacoes.csv


,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA
0,RJ,33,RJ - Largo do Bodegão,RJ0012,Rio de Janeiro,3304557.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
1,RJ,33,BR - São Bernardo,RJ0018,Belford Roxo,3300456.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
2,RJ,33,NI - Monteiro Lobato,RJ0019,Nova Iguaçu,3303500.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
3,RJ,33,RJ - Campo dos Afonsos,RJ0020,Rio de Janeiro,3304557.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
4,RJ,33,RJ - Taquara,RJ0021,Rio de Janeiro,3304557.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN



RJ_estacoes_enviada.xlsx


,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FINALIDADE,REP_ESPACIAL,INICIO,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS
0,RJ,33,BM - Boa Sorte,RJ0073,Barra Mansa,3300407,Referencia,Automática,Saint Gobain,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
1,RJ,33,BM - Sesi,RJ0074,Barra Mansa,3300407,Referencia,Automática,Saint Gobain,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
2,RJ,33,BM - Bocaininha,RJ0075,Barra Mansa,3300407,Referencia,Automática,Arcelormittal Barra Mansa,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
3,RJ,33,BM - Roberto Silveira,RJ0076,Barra Mansa,3300407,Referencia,Automática,Arcelormittal Barra Mansa,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
4,RJ,33,BM - Vista Alegre,RJ0077,Barra Mansa,3300407,Referencia,Automática,Arcelormittal Barra Mansa,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
